## Opay Bank Statement Parser

In [14]:
import pymupdf
import pandas as pd
import re

### Extracting and Loading Data

In [15]:
tables = []
with pymupdf.open(r"C:\Users\APIN PC\OneDrive\Documents\DS\circle_funds_externship\bank_statement_parser\opay_bs\opay_bankstatement.pdf") as doc:
    # print(len(doc))
    for page_number in range(len(doc)):
        page = doc.load_page(page_number)
        text_blocks = page.get_text('blocks')

        sorted_blocks = sorted(text_blocks, key=lambda b: b[1])

        table_data = []
        for block in sorted_blocks:
            # print(block)
            lines = block[4].split('\n')
            # print(lines)
            table_data.append(lines)

        if table_data:
            df = pd.DataFrame(table_data)
            processed_df = df
            # print(processed_df)

            if not processed_df.empty:
                # print(processed_df)
                tables.append(processed_df)

    if tables:
        concatenated_df = pd.concat(tables, ignore_index=True)
        bank_statement = concatenated_df
    else:
        bank_statement = pd.DataFrame()

In [16]:
bank_statement

,0,1,2,3,4,5,6,7
0,Account Statement,,None,None,None,None,None,None
1,Account Name,,None,None,None,None,None,None
2,VICTOR IYANUOLUWA AROWOSEGBE,,None,None,None,None,None,None
3,Currency,,None,None,None,None,None,None
4,Current Balance,,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...
1907,03 Apr 2025,OWealth Interest Earned,+0.09,1.07,E-Channel,250403997CmOUgwCGrTrJUmAflRcoi,,None
1908,23,,None,None,None,None,None,None
1909,2025 Apr 04 01:56:,,None,None,None,None,None,None
1910,33,04 Apr 2025,OWealth Interest Earned,+0.09,1.16,E-Channel,250404991nAUfbm7m2QQOyNRzOLQND,


In [17]:
# Renaming Columns
col = ['Trans.Time', 'Value Date', 'Description', 'Debit/Credit(#)', 'Balance(#)', 'Channel', 'Transaction Reference',
'NoneDrop']
bank_statement.columns = col
bank_statement

,Trans.Time,Value Date,Description,Debit/Credit(#),Balance(#),Channel,Transaction Reference,NoneDrop
0,Account Statement,,None,None,None,None,None,None
1,Account Name,,None,None,None,None,None,None
2,VICTOR IYANUOLUWA AROWOSEGBE,,None,None,None,None,None,None
3,Currency,,None,None,None,None,None,None
4,Current Balance,,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...
1907,03 Apr 2025,OWealth Interest Earned,+0.09,1.07,E-Channel,250403997CmOUgwCGrTrJUmAflRcoi,,None
1908,23,,None,None,None,None,None,None
1909,2025 Apr 04 01:56:,,None,None,None,None,None,None
1910,33,04 Apr 2025,OWealth Interest Earned,+0.09,1.16,E-Channel,250404991nAUfbm7m2QQOyNRzOLQND,


In [18]:
# Dropping multiple headers and rows with irrelevant values.
bs_df = bank_statement.copy()
bs_df = bs_df.drop(bs_df[bs_df['Trans.Time']=='Trans. Time'].index)
bs_df = bs_df[~bs_df['Trans.Time'].str.contains('^[A-Z]', regex=True)]
# bs_df = bs_df[~bs_df['Trans.Time'].str.contains('^₦', regex=True)] --  review
bs_df

,Trans.Time,Value Date,Description,Debit/Credit(#),Balance(#),Channel,Transaction Reference,NoneDrop
7,₦1.25,,None,None,None,None,None,None
13,"₦1,276,757.21",,None,None,None,None,None,None
14,"₦1,276,722.20",,None,None,None,None,None,None
15,9023134548,,None,None,None,None,None,None
19,319,,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...
1907,03 Apr 2025,OWealth Interest Earned,+0.09,1.07,E-Channel,250403997CmOUgwCGrTrJUmAflRcoi,,None
1908,23,,None,None,None,None,None,None
1909,2025 Apr 04 01:56:,,None,None,None,None,None,None
1910,33,04 Apr 2025,OWealth Interest Earned,+0.09,1.16,E-Channel,250404991nAUfbm7m2QQOyNRzOLQND,


In [19]:
date_pattern = re.compile(r"\d{2} [A-Za-z]{3} \d{4}$")
dt_complete_mapper = re.compile(r'^\d{4}\s(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s\d{2}\s\d{2}:\d{2}:\d{2}$')
# dt_incomplete_mapper = re.compile(r'^\d{4}\s(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s\d{2}\s\d{2}:\d{2}:$')

bs_df = bs_df.copy()
def check_datetime(row):
    cell_1 = str(row[0])
    return bool(
        re.match(date_pattern, cell_1)
        or
        re.match(dt_complete_mapper, cell_1))
        # or 
        # re.match(dt_incomplete_mapper, cell_1))

bs_df = bs_df[bs_df.apply(check_datetime, axis=1)]
bs_df.to_csv('open_second.csv')


C:\Users\APIN-PC\AppData\Local\Temp\ipykernel_28688\2649640046.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  cell_1 = str(row[0])


#### Tasks
- Shift rows with cell in column(Trans.Time) that matches [dd Month Year] format to the right.
- Delete Trans.Time column.
- Drop None values

#### Separating columns dataframe based on shifting goal

In [20]:
nas_dropped = bs_df.dropna(subset=['Balance(#)']).reset_index()
df_to_shift = nas_dropped[~(nas_dropped['Channel'] == 'E-Channel')].reset_index()
df_stable = nas_dropped[nas_dropped['Channel'] == 'E-Channel'].reset_index()

df_stable.info()
df_to_shift.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 202 entries, 0 to 201
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   level_0                202 non-null    int64 
 1   index                  202 non-null    int64 
 2   Trans.Time             202 non-null    object
 3   Value Date             202 non-null    object
 4   Description            202 non-null    object
 5   Debit/Credit(#)        202 non-null    object
 6   Balance(#)             202 non-null    object
 7   Channel                202 non-null    object
 8   Transaction Reference  202 non-null    object
 9   NoneDrop               201 non-null    object
dtypes: int64(2), object(8)
memory usage: 15.9+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 542 entries, 0 to 541
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   level_0             

### Performing shifting on dataframe selected.

In [21]:
shift_criteria = df_to_shift['Trans.Time'].str.contains('r"\b\d{2} [A-Za-z]{3} \d{4}\b"', regex=True).notna()
shift_criteria.index

for index in shift_criteria.index:
    index = int(index)
    df_to_shift.iloc[index,:] = df_to_shift.iloc[index,:].shift()
df_to_shift = df_to_shift

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\APIN-PC\AppData\Local\Temp\ipykernel_28688\2062310880.py:1: SyntaxWarning: invalid escape sequence '\d'
  shift_criteria = df_to_shift['Trans.Time'].str.contains('r"\b\d{2} [A-Za-z]{3} \d{4}\b"', regex=True).notna()


### Joining Both Transformed Dataframe together

In [22]:
final_df = pd.concat([df_stable,df_to_shift], axis=0)
final_df = final_df.drop(columns=['level_0', 'index', 'Trans.Time', 'NoneDrop']).reset_index()
final_df = final_df.drop(columns='index')
final_df

,Value Date,Description,Debit/Credit(#),Balance(#),Channel,Transaction Reference
0,12 Mar 2024,Spend & Save Withdrawal,"+2,394.69","2,394.69",E-Channel,240312014770779834
1,12 Mar 2024,OWealth Deposit(AutoSave),-294.69,0.00,E-Channel,240312145795962609
2,27 Mar 2024,Transfer from NWAGBO CHINEDUM OBIOMA,"+4,500.00","4,500.00",E-Channel,000004240327184135633860074199
3,12 Apr 2024,Transfer from EBENEZER AYOMIKUN FALODUN,"+10,000.00","10,000.00",E-Channel,000014240412184328237230179735
4,12 Apr 2024,OWealth Deposit(AutoSave),"-10,000.00",0.00,E-Channel,240412146767712953
...,...,...,...,...,...,...
739,30 Mar 2025,OWealth Interest Earned,+0.09,0.71,E-Channel,2503309971yubCWGb81ei38grs5BvL
740,31 Mar 2025,OWealth Interest Earned,+0.09,0.80,E-Channel,250331991NjIlD3hc3FB1cfDlCnNp1
741,01 Apr 2025,OWealth Interest Earned,+0.09,0.89,E-Channel,250401993FeW6pQKUwq8gwbX6pXZ39
742,02 Apr 2025,OWealth Interest Earned,+0.09,0.98,E-Channel,250402995dD0FEHSCceB5u4PBUOpUQ


In [23]:
final_df[final_df['Balance(#)'] == '--']

,Value Date,Description,Debit/Credit(#),Balance(#),Channel,Transaction Reference
283,01 Mar 2024,OWealth Interest Earned,+0.09,--,E-Channel,240301995IFn9H9nuWAJ0ECrwiqRCp
284,02 Mar 2024,OWealth Interest Earned,+0.09,--,E-Channel,240302996X4O1hI4B3EDylqovcrAFc
285,02 Mar 2024,OWealth Deposit(AutoSave),"+55,000.00",--,E-Channel,240302148774206305
286,03 Mar 2024,OWealth Interest Earned,+18.96,--,E-Channel,240303992K9T2sKy22f7fB0h8rt5T9
287,04 Mar 2024,OWealth Interest Earned,+18.97,--,E-Channel,24030499k9eC9avQBl7RIRwMWQt21
288,05 Mar 2024,OWealth Interest Earned,+18.98,--,E-Channel,240305990hy9YgjEBrxgcSMZDAs0B
289,06 Mar 2024,OWealth Interest Earned,+18.98,--,E-Channel,240306995AYDVrC2UyB0rltR2aCvLU
290,06 Mar 2024,Transfer to DIVINE GRACE AND MERCY MEDICINE STORE,"-10,000.00",--,E-Channel,100004240306091820112225032644
291,06 Mar 2024,Spend & Save Deposit,-500.00,--,E-Channel,240306144394846083
292,07 Mar 2024,OWealth Interest Earned,+15.37,--,E-Channel,240307992jl8C6Ru3NmobmqbEfIphE


### Assigning Correct Datatypes

In [24]:
final_df['Balance(#)'] = final_df['Balance(#)'].replace(r',', '', regex=True)
# final_df['Balance(#)'] = final_df['Balance(#)'].replace(r'--', '0', regex=True).astype(float)
final_df['Debit/Credit(#)'] = final_df['Debit/Credit(#)'].replace(r',', '', regex=True).astype(float)

final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 744 entries, 0 to 743
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Value Date             744 non-null    object 
 1   Description            744 non-null    object 
 2   Debit/Credit(#)        744 non-null    float64
 3   Balance(#)             744 non-null    object 
 4   Channel                744 non-null    object 
 5   Transaction Reference  744 non-null    object 
dtypes: float64(1), object(5)
memory usage: 35.0+ KB


### Creating new credit and debit columns 

In [25]:
final_df['Credit'] = round(final_df['Debit/Credit(#)'].apply(lambda x:x if x > 0 else 0), 2)
final_df['Debit'] = abs(round(final_df['Debit/Credit(#)'].apply(lambda x:x if x < 0 else 0), 2))
final_df.head(4)

,Value Date,Description,Debit/Credit(#),Balance(#),Channel,Transaction Reference,Credit,Debit
0,12 Mar 2024,Spend & Save Withdrawal,2394.69,2394.69,E-Channel,240312014770779834,2394.69,0.00
1,12 Mar 2024,OWealth Deposit(AutoSave),-294.69,0.00,E-Channel,240312145795962609,0.00,294.69
2,27 Mar 2024,Transfer from NWAGBO CHINEDUM OBIOMA,4500.00,4500.00,E-Channel,000004240327184135633860074199,4500.00,0.00
3,12 Apr 2024,Transfer from EBENEZER AYOMIKUN FALODUN,10000.00,10000.00,E-Channel,000014240412184328237230179735,10000.00,0.00


In [26]:
# Drop Debit/Credit Column
opay_clean = final_df.drop('Debit/Credit(#)', axis=1)

### Transform cleaned data to specified format.

In [ ]:
def transform_row_opay(row):
    return {
        'id':0,
        "type": "debit" if int(row["Debit"]) > 0 else "credit",
        "amount": row["Debit"] if int(row["Debit"]) > 0 else row["Credit"],
        "narration": row["Description"].replace(r"'", ""),
        "date": row["Value Date"],
        "balance": row["Balance(#)"] 
    }
transformed_opay = opay_clean.apply(transform_row_opay, axis=1).tolist()
transformed_opay_df = pd.DataFrame(transformed_opay)
transformed_opay_df.head(5)
transformed_opay_df.to_csv('Transformed_bs.csv')

### Convert to json

In [28]:
transformed_opay_df.to_json('transformed_opay.json', orient='records')

### Reviewing Profit/Loss Statement

In [29]:
transformed_opay_df.groupby('type')['amount'].sum()

credit = 1276721.75
debit = 1264022.21

diff = credit - debit
round(diff, 2)

12699.54

### Expected difference

In [30]:
round((credit - 1276757.21), 2)

-35.46

##### More work is to be done. 
Credit is correct. But Debit seems to be missing some values

`expected difference is = -35.46`

Check balance where value is 0.00